# LFS 2022 individual preprocessing: one table, two targets

<small>Assumes you've already gone through the exploratory data analysis: this notebook doesn't
redefine raw variable meanings (what `rel==1` means, what the `emps` codes are, etc.), it builds
on what that earlier work already covers.<br><br>
<b>Population</b>: rural, working-age (15+), employed, and in agriculture (`ind==10`). Everyone in
that population, not just wage-earners, so wage-earners are a subset of it. Restricted to
agriculture because that's what the rest of the project actually uses; see the note in step 3 for
why `ind` (industry) rather than `occ` (occupation) is the right definition, and why restricting
to agriculture here doesn't lose the household-context features (those come from the full,
unfiltered household roster, computed before this filter).<br><br>
<b>Two targets in the same table</b>:<br>
- `hrswk` (hours worked): populated for everyone. This is the <b>primary</b> outcome (see
`LITERATURE_outcome_metric_choice.md`: wage alone structurally can't see unpaid family
workers, who are disproportionately women; hours captures them).<br>
- `hourly_wage`: populated only where a wage exists (regular or irregular). This is the
<b>secondary</b> outcome, restricted to the paid subsample, same construction as before (division,
not sum; see `CLEANING_METHODOLOGY_EXPLAINED.md`).<br><br>
<b>Household-context features</b>, derived from this same individual file via `caseser`:
household size, number of other employed household members, and the head's own education/sex.
Computed directly rather than pulled from the household file's columns, since the household file
only ever tracks head+spouses and would reintroduce that blind spot.<br><br>
<b>Also pulled in from the household file itself</b>: `malinlf`/`feminlf` (household members 15+
in the labor force, by sex; a ready-made, fully-populated household-level participation measure)
and `occhd` (head's occupation, for parity with the individual file's `occ`).<br><br>
<b>Also</b>: `household_wage_total` and `wage_ratio` (spouse(s)' share of household wage income).
Irregular wages converted to monthly using each person's own real hours from the individual file,
not an external averages table. Covers <b>all</b> of the head's and spouse's wage income, any
industry, not agricultural income specifically. See step 2b for the full logic and a leakage
caveat that matters if this table gets used for wage/hours modeling later.<br><br>
<b>Also worth knowing</b>: `yeduc` (years of effective schooling) and `hlthins`/`socsec` (health
insurance / social security coverage) were already present in the raw individual file the whole
time. This script never drops columns, only rows, just under-documented until now.</small>

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent
LFS_DTA = REPO_ROOT / "Data" / "Labor Force Survey, LFS 2022 - Egypt, Arab Rep., 2022" / "Egypt 2022-LFS IND-V1.dta"
HH_DTA = REPO_ROOT / "Data" / "Labor Force Survey, LFS 2022 - Egypt, Arab Rep., 2022" / "Egypt 2022-LFS HH-V1.dta"
CROSSWALK_CSV = REPO_ROOT / "Data" / "crosswalk" / "reg_governorate_crosswalk.csv"
COMMAND_AREA_CROSSWALK_CSV = REPO_ROOT / "Data" / "crosswalk" / "command_area_governorate_crosswalk.csv"
SPATIAL_XLSX = REPO_ROOT / "Data" / "preprocessed_data_explain.xlsx"
OUTPUT_CSV = Path("lfs2022_individual_preprocessed.csv")

WEEKS_PER_MONTH = 365.25 / 7 / 12  # ~4.348
MIN_COMMAND_AREA_COVERAGE = 0.05
MIN_WORKING_AGE = 15

IND_LABELS = {10: "Agriculture, forestry and fishing", 20: "Mining and quarrying", 30: "Manufacturing",
              40: "Electricity, gas and water supply", 50: "Construction", 60: "Wholesale and retail trade",
              70: "Transportation and storage", 80: "Accommodation and food service activities",
              90: "Information and communication", 100: "Financial and insurance activities",
              110: "Real estate, professional and support service activities", 120: "Public administration and defense",
              130: "Education", 140: "Human health and social work activities", 150: "Other activities"}
EMPS_LABELS = {1: "Employee", 2: "Employer", 3: "Own-account/self-employed",
               4: "Unpaid family worker", 5: "Producers cooperative", 6: "Not classifiable"}
EDUC_LABELS = {1: "None", 2: "Primary/Lower secondary", 3: "Secondary",
               4: "Post secondary or equivalent", 5: "University", 6: "Postgraduate"}
EMPSTAB_LABELS = {1: "Full time/Regular", 2: "Part time/Temporary", 3: "Seasonal/Irregular"}
REL_LABELS = {1: "Head", 2: "Spouse", 3: "Son/daughter", 4: "Parent", 5: "Sibling",
              6: "Grandchild", 7: "Other relative", 8: "Non-relative"}
OCC_LABELS = {10: "Managers", 20: "Professionals", 30: "Technicians", 40: "Clerks", 50: "Service/sales",
              60: "Skilled agricultural/fishery", 70: "Craft workers", 80: "Plant/machine operators",
              90: "Elementary occupations", 100: "Armed forces", 998: "Other", 999: "Not stated"}
# Standard ERF-harmonized marital-status scheme (code 1 confirmed as "Never married" in the
# dictionary; 2-5 follow the standard ILO/ERF ordering, not independently re-verified against a
# complete codebook; the source dictionary's value list is truncated for this variable).
MART_LABELS = {1: "Never married", 2: "Married", 3: "Widowed", 4: "Divorced", 5: "Separated"}
YES_NO = {0: "No", 1: "Yes"}

### 1. Load the full individual file, plus the household file for a few household-level features (we need the whole thing before filtering, for the household-enrichment step below)

In [2]:
df = pd.read_stata(LFS_DTA, convert_categoricals=False)
hh = pd.read_stata(HH_DTA, convert_categoricals=False)
print("individual shape:", df.shape)
print("household shape:", hh.shape)

individual shape: (299423, 106)
household shape: (76976, 116)


`yeduc`, `hlthins`, `socsec` are already in `df`: this script never drops columns, only rows, so
they've been carried through the whole time. Confirming they're actually populated for the
population we care about:

In [3]:
for c in ["yeduc", "hlthins", "socsec"]:
    print(f"{c}: {df[c].notna().sum():,} non-null of {len(df):,}")

yeduc: 260,591 non-null of 299,423
hlthins: 76,411 non-null of 299,423
socsec: 76,411 non-null of 299,423


### 2. Household-context features: most computed from this same file via `caseser`, a few pulled directly from the household file

<small>Done on the <b>full, unfiltered</b> file, so household size and composition reflect the whole
household (kids included), not just whoever happens to pass our later age/employment filters.
`malinlf`/`feminlf`/`occhd` come straight from the household file: one row per `caseser` there, so
it's a clean join, no aggregation needed.</small>

In [4]:
household_size = df.groupby("caseser").size()
n_employed_in_hh = df[df["emps"].notna()].groupby("caseser").size()
head_info = df[df["rel"] == 1].drop_duplicates("caseser").set_index("caseser")[["educ", "sex"]]
head_info.columns = ["head_educ", "head_sex"]

print("household_size: computed for", len(household_size), "households")
print("n_employed_in_hh: computed for", len(n_employed_in_hh), "households (that have >=1 employed member)")
print("head_info: found a head for", len(head_info), "households")
print("hh file: malinlf/feminlf/occhd available for", hh["caseser"].nunique(), "households")

household_size: computed for 76976 households
n_employed_in_hh: computed for 57695 households (that have >=1 employed member)
head_info: found a head for 76976 households
hh file: malinlf/feminlf/occhd available for 76976 households


### 2b. Household wage total and head:spouse wage ratio, using each person's own real hours

<small>Irregular (seasonal/daily) wages get converted to monthly using each person's own real
`hrswk`, matched via `caseser`+`rel`, not an external averages table. Falls back to an
industry-average computed from this file's own data only when no individual match exists.
`wage_ratio` = spouse(s)' share of household wage income (`wagesp / (wagehd + wagesp)`), filled to
0.5 where neither earns anything. Computed on rural households only, matching the rest of this
notebook, rather than the full national file.<br><br>
<b>Scope</b>: `household_wage_total` and `wage_ratio` cover the head's and spouse's wage income in
whatever industry they work in, not agricultural income specifically. A household's head could
work in construction while the row this gets attached to is their agricultural-worker
child.<br><br>
<b>Leakage caveat, matters for modeling later</b>: for a row where the individual <i>is</i> the
household head or a spouse, these two columns are partly built from that same person's own
`totwag`/`irrgwag`, the values `hourly_wage` is built from. Using them as predictive features for
that person's own wage or hours would be circular. Check `rel_label` before using either column
that way; sons/daughters/other relatives aren't in `wagehd`/`wagesp` at all, so no leakage for
them.<br><br>
Known limitation: this only ever compares the head's income to the spouse(s)' income. Wage-earners
who are neither (sons, daughters, other relatives) aren't represented in this ratio, kept anyway
since it's still a useful bargaining-power signal for the head/spouse population it does
cover.</small>

In [5]:
avg_hours_by_ind = df[df["hrswk"] > 0].groupby("ind")["hrswk"].mean()

head_hours = df[df["rel"] == 1].drop_duplicates("caseser").set_index("caseser")["hrswk"]
spouses_ind = df[df["rel"] == 2].sort_values(["caseser", "pnum"]).copy()
spouses_ind["spouse_order"] = spouses_ind.groupby("caseser").cumcount() + 1
spouse_hours = {n: spouses_ind[spouses_ind["spouse_order"] == n].set_index("caseser")["hrswk"] for n in [1, 2, 3]}

def convert_irregular(daily_wage, ind_code, real_hours, label):
    used_real = real_hours.notna() & (real_hours > 0)
    hours = real_hours.where(used_real, ind_code.map(avg_hours_by_ind))
    n_earners = (daily_wage > 0).sum()
    n_real = (used_real & (daily_wage > 0)).sum()
    if n_earners:
        print(f"  {label}: {n_real}/{n_earners} irregular earners matched to real individual hours "
              f"({n_real/n_earners*100:.1f}%), rest used industry average")
    workdays_per_week = hours / 8
    return daily_wage * workdays_per_week * 4

hh_wage = hh[hh["rururb"] == 0].copy()
hh_wage["head_hrswk"] = hh_wage["caseser"].map(head_hours)
print("Irregular-wage-to-monthly conversion, real-hours match rate (rural households only):")
hh_wage["irrwagehd"] = convert_irregular(hh_wage["irrgwaghd"].fillna(0), hh_wage["indhd"], hh_wage["head_hrswk"], "head")
for n in [1, 2, 3]:
    hh_wage[f"sp_{n}_hrswk"] = hh_wage["caseser"].map(spouse_hours[n])
    hh_wage[f"irrwagesp_{n}"] = convert_irregular(
        hh_wage[f"irrgwagsp_{n}"].fillna(0), hh_wage[f"indsp_{n}"], hh_wage[f"sp_{n}_hrswk"], f"spouse_{n}"
    )

hh_wage["wagehd"] = hh_wage[["empinchd", "sempinchd", "totwaghd", "irrwagehd"]].sum(axis=1, skipna=True)
sp_cols = []
for n in [1, 2, 3]:
    sp_cols += [f"empincsp_{n}", f"sempincsp_{n}", f"totwagsp_{n}", f"irrwagesp_{n}"]
hh_wage["wagesp"] = hh_wage[sp_cols].sum(axis=1, skipna=True)
hh_wage["household_wage_total"] = hh_wage["wagehd"] + hh_wage["wagesp"]
hh_wage["wage_ratio"] = (hh_wage["wagesp"] / hh_wage["household_wage_total"]).fillna(0.5)

print()
print(hh_wage[["household_wage_total", "wage_ratio"]].describe())

Irregular-wage-to-monthly conversion, real-hours match rate (rural households only):
  head: 7511/7538 irregular earners matched to real individual hours (99.6%), rest used industry average
  spouse_1: 195/200 irregular earners matched to real individual hours (97.5%), rest used industry average

       household_wage_total    wage_ratio
count          45426.000000  45426.000000
mean            2209.675539      0.190799
std             3359.425282      0.252778
min                0.000000      0.000000
25%                0.000000      0.000000
50%             2250.000000      0.000000
75%             3200.000000      0.500000
max           150000.000000      1.000000


### 3. Filter to working-age, rural, employed, agricultural individuals: the analysis population

<small>Agriculture defined by `ind==10` (industry/economic activity), not `occ` (job title).
Checked against the occupation-based definition: industry gives a bigger sample (14,006 vs. 13,433
in this population) and doesn't wrongly drop real agricultural workers whose job title isn't coded
"skilled agricultural worker" (farm managers, elementary farm laborers, machine operators). Matches
the definition used in the closest literature precedents (Afridi, Mahajan & Sangwan 2022; Van den
Broeck et al. 2023).<br><br>
The household-context features above were computed on the full, unfiltered file, so restricting
these rows to agriculture doesn't lose any of that household information: it was never dependent
on keeping non-agricultural household members' own rows around.</small>

In [6]:
n_underage = (df["age"] < MIN_WORKING_AGE).sum()
working_age = df[df["age"] >= MIN_WORKING_AGE].copy()
print(f"Dropped {n_underage} respondents under age {MIN_WORKING_AGE} (working-age floor)")

rural = working_age[working_age["rururb"] == 0].copy()
print("shape after rural filter:", rural.shape)

employed = rural[rural["emps"].notna()].copy()
print("shape after employed filter:", employed.shape)
print(employed["emps"].map(EMPS_LABELS).value_counts())

agri = employed[employed["ind"] == 10].copy()
print("shape after agriculture filter (ind==10):", agri.shape)
print(f"({(agri['sex']==1).sum()} male, {(agri['sex']==2).sum()} female)")

Dropped 100476 respondents under age 15 (working-age floor)
shape after rural filter: (116154, 106)
shape after employed filter: (46027, 106)
emps
Employee                     32521
Own-account/self-employed     9375
Unpaid family worker          3068
Employer                      1032
Name: count, dtype: int64
shape after agriculture filter (ind==10): (14043, 106)
(12019 male, 2024 female)


### 4. Attach the household-context features to each person's row

In [7]:
agri["household_size"] = agri["caseser"].map(household_size)
agri["n_other_employed"] = agri["caseser"].map(n_employed_in_hh) - 1  # exclude self
agri = agri.merge(head_info, on="caseser", how="left")
agri = agri.merge(hh[["caseser", "malinlf", "feminlf", "occhd"]], on="caseser", how="left")
agri = agri.merge(hh_wage[["caseser", "household_wage_total", "wage_ratio"]], on="caseser", how="left")

agri[["caseser","rel","household_size","n_other_employed","head_educ","head_sex",
      "malinlf","feminlf","occhd","household_wage_total","wage_ratio"]].sample(5, random_state=1)

,caseser,rel,household_size,n_other_employed,head_educ,head_sex,malinlf,feminlf,occhd,household_wage_total,wage_ratio
272,1.220162e+10,1,4,1,1.0,1,2,0,60.0,2000.0,0.0
4300,1.505043e+11,1,2,1,1.0,1,1,1,10.0,1300.0,0.0
9465,2.304172e+11,1,5,0,2.0,1,1,0,60.0,5100.0,0.0
1060,2.112303e+10,1,5,0,1.0,1,1,0,60.0,2500.0,0.0
588,1.513043e+10,2,7,1,2.0,1,1,1,60.0,1000.0,0.0


### 5. Primary target: hours worked (no conversion needed, already one consistent unit)

In [8]:
valid_hours = agri["hrswk"].notna() & (agri["hrswk"] > 0)
n_dropped = (~valid_hours).sum()
final = agri[valid_hours].copy()
print(f"Dropped {n_dropped} people with missing/zero hours worked")
print(f"Remaining: {len(final)} ({(final['sex']==1).sum()} male, {(final['sex']==2).sum()} female)")

Dropped 37 people with missing/zero hours worked


Remaining: 14006 (11993 male, 2013 female)


### 6. Secondary target: hourly wage, populated only for the wage-earning subset

<small>Same construction as before: division (unit conversion), not sum or average. `TOTWAG` and
`IRRGWAG` are confirmed mutually exclusive.</small>

In [9]:
overlap = ((final["totwag"] > 0) & (final["irrgwag"] > 0)).sum()
assert overlap == 0, f"{overlap} respondents have both totwag and irrgwag > 0, unexpected"

is_regular = final["totwag"] > 0
is_irregular = final["irrgwag"] > 0
is_wage_earner = is_regular | is_irregular

final["wage_type"] = np.select([is_regular, is_irregular], ["regular", "irregular"], default=None)
final["hourly_wage"] = np.where(
    is_regular, final["totwag"] / (final["hrswk"] * WEEKS_PER_MONTH),
    np.where(is_irregular, final["irrgwag"] / (final["hrswk"] / 7), np.nan)
)

print(f"Full population (primary, hours): {len(final)}")
print(f"Wage-earning subset (secondary, wage): {is_wage_earner.sum()} "
      f"({(is_wage_earner & (final['sex']==2)).sum()} female)")

Full population (primary, hours): 14006
Wage-earning subset (secondary, wage): 7152 (363 female)


### 7. Labels and flags

<small>`empcont` (employment contract type) is deliberately left unlabeled: the dictionary only
documents code 200 ("No contract"); the data also contains codes 125 and 130 with no confirmed
definition in any file available here. Rather than guess, it stays as a raw code.</small>

In [10]:
final["sex_label"] = final["sex"].map({1: "Male", 2: "Female"})
final["educ_label"] = final["educ"].map(EDUC_LABELS)
final["ind_label"] = final["ind"].map(IND_LABELS)
final["emps_label"] = final["emps"].map(EMPS_LABELS)
final["mart_label"] = final["mart"].map(MART_LABELS)
final["empstab_label"] = final["empstab"].map(EMPSTAB_LABELS)
final["rel_label"] = final["rel"].map(REL_LABELS)
final["occhd_label"] = final["occhd"].map(OCC_LABELS)
final["lit_label"] = final["lit"].map(YES_NO)
final["hlthins_label"] = final["hlthins"].map(YES_NO)
final["socsec_label"] = final["socsec"].map(YES_NO)
final["is_unpaid_family"] = final["emps"] == 4
final["is_wage_employee"] = final["emps"] == 1

### 8. Attach governorate-level spatial covariates and the command-area relevance filter

In [11]:
crosswalk = pd.read_csv(CROSSWALK_CSV)
spatial = pd.read_excel(SPATIAL_XLSX)

merged = final.merge(crosswalk[["reg_code", "OBJECTID", "NAME1_"]], left_on="reg", right_on="reg_code", how="left")
n_unmatched = merged["OBJECTID"].isna().sum()
if n_unmatched:
    names = merged.loc[merged["OBJECTID"].isna(), "reg"].map(dict(zip(crosswalk["reg_code"], crosswalk["reg_name_lfs"]))).unique()
    print(f"Note: {n_unmatched} respondents in governorates without RIBASIM-linked biophysical data ({list(names)})")

merged = merged.merge(spatial, on="OBJECTID", how="left", suffixes=("", "_spatial"))

ca_crosswalk = pd.read_csv(COMMAND_AREA_CROSSWALK_CSV)
coverage = ca_crosswalk.groupby("gov_NAME1_")["pct_of_governorate_area"].sum()
merged["command_area_coverage"] = merged["NAME1_"].map(coverage).fillna(0.0)

n_before = len(merged)
n_female_before = (merged["sex"] == 2).sum()
n_wage_female_before = ((merged["hourly_wage"].notna()) & (merged["sex"] == 2)).sum()
result = merged[merged["command_area_coverage"] >= MIN_COMMAND_AREA_COVERAGE].copy()
dropped_govs = sorted(n for n in merged.loc[merged["command_area_coverage"] < MIN_COMMAND_AREA_COVERAGE, "NAME1_"].unique() if pd.notna(n))

print(f"{n_before} -> {len(result)} rows ({n_female_before} -> {(result['sex']==2).sum()} female)")
print(f"Wage-earning women: {n_wage_female_before} -> {((result['hourly_wage'].notna()) & (result['sex']==2)).sum()}")
print("Governorates dropped:", dropped_govs)

Note: 87 respondents in governorates without RIBASIM-linked biophysical data (['El-wadi El-Gidid'])
14006 -> 13264 rows (2013 -> 2010 female)
Wage-earning women: 363 -> 362
Governorates dropped: ['Aswan', 'Giza', 'Matrouh', 'South Sinai']


### 9. Save

In [12]:
result.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(result)} rows to {OUTPUT_CSV}")
print()
print(f"PRIMARY (hours worked), full population: {len(result)} "
      f"({(result['sex']==1).sum()} male, {(result['sex']==2).sum()} female)")
wage_sub = result[result["hourly_wage"].notna()]
print(f"SECONDARY (hourly wage), paid subsample: {len(wage_sub)} "
      f"({(wage_sub['sex']==1).sum()} male, {(wage_sub['sex']==2).sum()} female)")

Wrote 13264 rows to lfs2022_individual_preprocessed.csv

PRIMARY (hours worked), full population: 13264 (11254 male, 2010 female)
SECONDARY (hourly wage), paid subsample: 6676 (6314 male, 362 female)
